# Music Library Exploration & Validation

This notebook explores the MTG-Jamendo dataset, validates the curated music index,
and visualizes the tag/emotion distribution.

## Setup

1. Clone the MTG-Jamendo metadata:
   ```bash
   git clone https://github.com/MTG/mtg-jamendo-dataset.git /tmp/mtg-jamendo
   ```

2. Build the music index:
   ```bash
   uv run python scripts/build_music_index.py /tmp/mtg-jamendo/data
   ```

3. Then run this notebook to explore and validate.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from lectoria.services.music import (
    TAG_VOCABULARY,
    TAG_TO_EMOTION,
    EMOTION_TO_TAGS,
    load_music_index,
    scene_to_vector,
    _cosine_similarity,
    tags_to_vector,
)
from lectoria.models.ncm import Emotion, Pacing, SceneType

## 1. Tag Vocabulary Overview

In [ ]:
print(f"Total Jamendo mood/theme tags: {len(TAG_VOCABULARY)}")
print()

# Tags per emotion
emotion_mapped = {tag: str(e) for tag, e in TAG_TO_EMOTION.items() if e is not None}
unmapped = [tag for tag, e in TAG_TO_EMOTION.items() if e is None]

print(f"Emotion-mapped tags: {len(emotion_mapped)}")
print(f"Unmapped tags (generic/use-case): {len(unmapped)} -> {unmapped}")
print()

# Distribution of tags per emotion
tags_per_emotion = Counter(emotion_mapped.values())
for emotion in sorted(tags_per_emotion.keys()):
    tags = [t for t, e in emotion_mapped.items() if e == emotion]
    print(f"  {emotion:12s} ({len(tags):2d} tags): {', '.join(sorted(tags))}")

## 2. Load & Validate Music Index

In [ ]:
index_path = project_root / "data" / "music" / "music_index.json"

if not index_path.exists():
    print(f"Music index not found at {index_path}")
    print("Run: uv run python scripts/build_music_index.py /path/to/mtg-jamendo/data")
else:
    index = load_music_index(index_path)
    print(f"Loaded {len(index)} tracks")
    print()

    # Emotion distribution
    emotion_counts = Counter(t.emotion_primary for t in index)
    print("Tracks per emotion:")
    for emotion in sorted(emotion_counts.keys()):
        count = emotion_counts[emotion]
        bar = "#" * (count // 2)
        print(f"  {emotion:12s}: {count:4d} {bar}")
    print(f"  {'TOTAL':12s}: {len(index):4d}")
    print()

    # Duration stats
    durations = [t.duration_seconds for t in index]
    print(
        f"Duration: min={min(durations):.0f}s, max={max(durations):.0f}s, "
        f"mean={np.mean(durations):.0f}s, median={np.median(durations):.0f}s"
    )
    print()

    # Tags per track stats
    tag_counts = [len(t.tags) for t in index]
    print(
        f"Tags/track: min={min(tag_counts)}, max={max(tag_counts)}, "
        f"mean={np.mean(tag_counts):.1f}, median={np.median(tag_counts):.0f}"
    )

    # Coverage check: at least 3 per emotion?
    print()
    all_emotions = {e.value for e in Emotion}
    for e in sorted(all_emotions):
        c = emotion_counts.get(e, 0)
        status = "OK" if c >= 3 else "!! BELOW MINIMUM"
        print(f"  {e:12s}: {c:4d} tracks [{status}]")

## 3. Scene Vector Examples

Visualize what tag vectors look like for different scene attribute combinations.

In [ ]:
from lectoria.models.ncm import Scene

examples = [
    ("Tense action scene (fast)", Emotion.TENSION, Pacing.FAST, SceneType.ACTION),
    ("Peaceful description (slow)", Emotion.PEACE, Pacing.SLOW, SceneType.DESCRIPTION),
    ("Joyful dialogue (medium)", Emotion.JOY, Pacing.MEDIUM, SceneType.DIALOGUE),
    ("Mysterious introspection (slow)", Emotion.MYSTERY, Pacing.SLOW, SceneType.INTROSPECTION),
    ("Romantic dialogue (medium)", Emotion.ROMANCE, Pacing.MEDIUM, SceneType.DIALOGUE),
    ("Exciting transition (fast)", Emotion.EXCITEMENT, Pacing.FAST, SceneType.TRANSITION),
]

for label, emotion, pacing, scene_type in examples:
    scene = Scene(
        scene_index=1,
        start_paragraph=1,
        end_paragraph=10,
        emotion=emotion,
        pacing=pacing,
        scene_type=scene_type,
    )
    vec = scene_to_vector(scene)
    active_tags = [TAG_VOCABULARY[i] for i, v in enumerate(vec) if v > 0]
    print(f"{label}")
    print(f"  Active tags ({len(active_tags)}): {', '.join(active_tags)}")
    print()

## 4. Matching Simulation

Test how matching works with synthetic scenes against the index.

In [ ]:
from lectoria.services.music import match_scene_to_track_detailed

if index_path.exists():
    for label, emotion, pacing, scene_type in examples:
        scene = Scene(
            scene_index=1,
            start_paragraph=1,
            end_paragraph=10,
            emotion=emotion,
            pacing=pacing,
            scene_type=scene_type,
        )
        result = match_scene_to_track_detailed(scene, index, top_n=3)
        print(f"{label}")
        print(f"  Selected: {result['selected_track']} (score: {result['score']:.3f})")
        if result["fell_back_to_full_index"]:
            print(f"  !! Fell back to full index (no tracks for {emotion})")
        for i, c in enumerate(result["candidates"]):
            print(f"    #{i + 1} {c['track_id']} score={c['score']:.3f} tags={c['tags']}")
        print()
else:
    print("Music index not built yet. Run build_music_index.py first.")

## 5. Inter-Emotion Similarity Matrix

Check how distinct the tag vectors are between emotion categories.
We want low similarity between different clusters and moderate within.

In [ ]:
emotions = list(Emotion)
emotion_vecs = {}

for e in emotions:
    tags = EMOTION_TO_TAGS.get(e, [])
    emotion_vecs[e] = tags_to_vector(tags)

print(f"{'':12s}", end="")
for e in emotions:
    print(f"{e.value:>10s}", end="")
print()

for e1 in emotions:
    print(f"{e1.value:12s}", end="")
    for e2 in emotions:
        sim = _cosine_similarity(emotion_vecs[e1], emotion_vecs[e2])
        print(f"{sim:10.2f}", end="")
    print()